In [2]:
import re, time, random
from typing import List, Dict, Optional
from urllib.parse import urlparse, parse_qs

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from geopy.geocoders import Nominatim


In [3]:
class ImmoweltScraper:
    """Immowelt scraper using HTTP requests - fast and reliable!"""

    def __init__(self, curl_command: str = None, base_url: str = None, 
                 headers: Dict = None, cookies: Dict = None, delay_range: tuple = (3, 6)):
        self.delay_range = delay_range
        self.scraped_data = []

        # Option 1: Parse from cURL command (recommended - includes auth)
        if curl_command:
            config = self._parse_curl(curl_command)
            self.base_url = config['base_url']
            self.params = config['params']
            
            self.session = requests.Session()
            self.session.headers.update(config['headers'])
            self.session.cookies.update(config['cookies'])
            
            print(f"✓ Initialized from cURL with {len(config['cookies'])} cookies, {len(config['headers'])} headers")
        
        # Option 2: Simple URL template (for basic usage without auth)
        elif base_url:
            parsed_url = urlparse(base_url.replace('{page}', '1'))
            self.base_url = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}"
            self.params = {k: v[0] for k, v in parse_qs(parsed_url.query).items()}
            
            self.session = requests.Session()
            if headers:
                self.session.headers.update(headers)
            if cookies:
                self.session.cookies.update(cookies)
            
            print(f"✓ Initialized from URL template")
        
        else:
            raise ValueError("Must provide either curl_command or base_url")

    def _parse_curl(self, cmd: str) -> Dict:
        """Parse cURL command to extract URL, headers, and cookies"""
        c = cmd.replace('\\\n', ' ').strip()

        # Extract URL
        url = re.search(r"curl\s+['\"]?([^'\"]+?)['\"]?\s+(?:-|$)", c).group(1)

        # Extract headers (skip cookie header)
        headers = {}
        for m in re.finditer(r"-H\s+['\"]([^:]+):\s*([^'\"]+)['\"]", c):
            if m.group(1).lower() != 'cookie':
                headers[m.group(1).strip()] = m.group(2).strip()

        # Extract cookies from both -b flag and -H cookie header
        cookies = {}
        for pattern in [r"(?:-b|--cookie)\s+['\"](.+?)['\"](?:\s|$)",
                       r"-H\s+['\"]cookie:\s*([^'\"]+)['\"]"]:
            m = re.search(pattern, c, re.I)
            if m:
                for pair in m.group(1).split(';'):
                    if '=' in pair:
                        k, v = pair.split('=', 1)
                        cookies[k.strip()] = v.strip()

        # Parse URL components
        parsed_url = urlparse(url)
        return {
            'base_url': f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}",
            'params': {k: v[0] for k, v in parse_qs(parsed_url.query).items()},
            'headers': headers,
            'cookies': cookies
        }

    def scrape_page(self, page_num: int) -> Optional[str]:
        """Fetch a single page HTML"""
        params = self.params.copy()
        params['page'] = str(page_num)

        try:
            response = self.session.get(self.base_url, params=params, timeout=30)
            return response.text if response.status_code == 200 else None
        except Exception as e:
            print(f"Error page {page_num}: {e}")
            return None

    def extract_data(self, html: str, page_num: int) -> List[Dict]:
        """Extract listing data from HTML"""
        soup = BeautifulSoup(html, 'lxml')
        links = soup.select('a[data-testid="card-mfe-covering-link-testid"]')
        images = soup.select('img[alt]')  # Images with alt text

        listings = []
        for i, link in enumerate(links):
            try:
                detail_url = link.get('href', '')
                title_attr = link.get('title', '')

                # Find matching image by looking for property info in alt text
                alt_text = ''
                for img in images:
                    alt = img.get('alt', '')
                    # Check if this image's alt text contains property info
                    if alt and any(word in alt for word in ['Wohnung', 'Haus', 'Zimmer']):
                        alt_text = alt
                        images.remove(img)  # Remove so next listing gets next image
                        break

                # Use alt_text if available, otherwise fall back to title
                info_str = alt_text if alt_text else title_attr

                listings.append({
                    'detail_url': detail_url,
                    'raw_info': info_str
                })
            except Exception as e:
                print(f"Error parsing a listing: {e}")

        return listings

    def detect_total_pages(self, html: str) -> int:
        """Auto-detect total number of pages from pagination nav"""
        soup = BeautifulSoup(html, 'lxml')
        
        # Find pagination nav and look for page number buttons
        nav = soup.find('nav', {'aria-label': 'pagination navigation'})
        if nav:
            # Find all buttons with aria-label like "zu seite X"
            page_numbers = []
            for button in nav.find_all('button'):
                aria_label = button.get('aria-label', '')
                # Match patterns like "zu seite 64" or "aktuelle seite, seite 1"
                match = re.search(r'seite\s+(\d+)', aria_label, re.I)
                if match:
                    page_numbers.append(int(match.group(1)))
            
            # Return the highest page number found
            if page_numbers:
                return max(page_numbers)
        
        # Fallback to 65 if detection fails
        return 4

    def save_data(self, filename: str, fmt: str = "csv"):
        """Save scraped data to file"""
        if not self.scraped_data:
            print("No data to save")
            return

        df = pd.DataFrame(self.scraped_data)
        if fmt.lower() == "csv":
            df.to_csv(filename, index=False)
        elif fmt.lower() == "json":
            df.to_json(filename, orient="records", indent=2)
        elif fmt.lower() == "excel":
            df.to_excel(filename, index=False)

        print(f"✓ Saved {len(df)} listings to {filename}")

    def run_scraper(self, start_page: int = 1, end_page: Optional[int] = None):
        """Main scraping loop with progress bar"""
        # Store start/end for filename
        self.start_page = start_page
        
        # Fetch first page
        first_page_html = self.scrape_page(start_page)
        if not first_page_html:
            print("Failed to fetch first page!")
            return

        # Auto-detect total pages
        if end_page is None:
            end_page = self.detect_total_pages(first_page_html)
            print(f"Auto-detected {end_page} pages")
        
        self.end_page = end_page

        # Extract first page
        self.scraped_data.extend(self.extract_data(first_page_html, start_page))

        # Scrape remaining pages with progress bar
        for page in tqdm(range(start_page + 1, end_page + 1), desc="Scraping"):
            time.sleep(random.uniform(*self.delay_range))
            html = self.scrape_page(page)
            if html:
                self.scraped_data.extend(self.extract_data(html, page))

        print(f"\n✓ Scraped {len(self.scraped_data)} total listings")

    def close(self):
        """Close HTTP session"""
        self.session.close()

print("✓ ImmoweltScraper class defined")

✓ ImmoweltScraper class defined


In [4]:
# ===== OPTION 1: Use cURL Command (Recommended - includes authentication) =====
# Paste your cURL command from Chrome DevTools here
CURL_COMMAND = """

"""

# Initialize with cURL
scraper = ImmoweltScraper(curl_command=CURL_COMMAND, delay_range=(3, 6))


# ===== OPTION 2: Simple URL Template (Basic usage - may get blocked without cookies) =====
# Uncomment this section to use simple URL template instead:

# base_url_template = (
#     "https://www.immowelt.de/classified-search"
#     "?distributionTypes=Rent"
#     "&estateTypes=House,Apartment"
#     "&locations=AD08DE8634"
#     "&projectTypes=New_Build,Flatsharing,Stock"
#     "&page={page}"
# )
# 
# scraper = ImmoweltScraper(base_url=base_url_template, delay_range=(3, 6))

AttributeError: 'NoneType' object has no attribute 'group'

In [ ]:
# Run scraper (auto-detects ~64 pages)
start_page = 1
end_page = None  # None = auto-detect, or specify number like 4 for testing

scraper.run_scraper(start_page=start_page, end_page=end_page)

# Generate filename based on actual pages scraped
output_filename = f"immowelt_pages_{scraper.start_page}_{scraper.end_page}.csv"
scraper.save_data(output_filename, fmt="csv")

# Cleanup
scraper.close()

print(f"\n✅ Done! Saved to {output_filename}")

In [12]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from geopy.geocoders import Nominatim
import time

# 1. BUILD ADDRESS COLUMN
def build_address(df, street_col, house_num_col, postal_col, neigh_col, city_col):
    """
    Combines address parts into a single address string per row.
    """
    df['address'] = (
        df[street_col].fillna('').astype(str) + ' ' +
        df[house_num_col].fillna('').astype(str) + ', ' +
        df[postal_col].fillna('').astype(str) + ' ' +
        df[neigh_col].fillna('').astype(str) + ' ' +
        df[city_col].fillna('').astype(str)
    ).str.strip()
    return df


data = pd.read_csv('parsed_listings.csv')
data = data.rename(columns={'city_region': 'neighborhood'})

data['postal_code'] = data['postal_code'].astype('Int64').astype(str)
data


,detail_url,raw_info,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,neighborhood,postal_code,region,city,first_tenant,type_normalized
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung
2,https://www.immowelt.de/expose/a9bef6a3-1c32-4...,"Wohnung zur Miete - Erstbezug 1.980 € 3,5 Zimm...",Wohnung,1980.0,3.5,90.3,0,EG Archenholdstraße,21,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,Friedenau Berlin,12159,Friedenau,Berlin,no,Wohnung
4,https://www.immowelt.de/expose/324a0641-c80f-4...,Wohnung zur Miete 2.860 € 4 Zimmer 102 m² EG f...,Wohnung,2860.0,4.0,102.0,0,EG Herderstraße,5,Charlottenburg Berlin,10625,Charlottenburg,Berlin,no,Wohnung
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,Spandau Berlin,13591,Spandau,Berlin,no,Studio
5340,https://www.immowelt.de/expose/5a4ba852-5719-4...,Wohnungsswap.de,Wohnung,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,no,Wohnung
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,Kreuzberg Berlin,10997,Kreuzberg,Berlin,no,Studio
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,Kaulsdorf Berlin,12627,Kaulsdorf,Berlin,yes,Studio


In [13]:
data = build_address(
    data,
    street_col='street',
    house_num_col='house_number',
    postal_col='postal_code',
    neigh_col='region',
    city_col='city'
)

In [14]:
data

,detail_url,raw_info,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,neighborhood,postal_code,region,city,first_tenant,type_normalized,address
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung,", 10315 Friedrichsfelde Berlin"
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung,", 10315 Friedrichsfelde Berlin"
2,https://www.immowelt.de/expose/a9bef6a3-1c32-4...,"Wohnung zur Miete - Erstbezug 1.980 € 3,5 Zimm...",Wohnung,1980.0,3.5,90.3,0,EG Archenholdstraße,21,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung,"EG Archenholdstraße 21, 10315 Friedrichsfelde..."
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,Friedenau Berlin,12159,Friedenau,Berlin,no,Wohnung,", 12159 Friedenau Berlin"
4,https://www.immowelt.de/expose/324a0641-c80f-4...,Wohnung zur Miete 2.860 € 4 Zimmer 102 m² EG f...,Wohnung,2860.0,4.0,102.0,0,EG Herderstraße,5,Charlottenburg Berlin,10625,Charlottenburg,Berlin,no,Wohnung,"EG Herderstraße 5, 10625 Charlottenburg Berlin"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,Spandau Berlin,13591,Spandau,Berlin,no,Studio,", 13591 Spandau Berlin"
5340,https://www.immowelt.de/expose/5a4ba852-5719-4...,Wohnungsswap.de,Wohnung,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,no,Wohnung,", <NA>"
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,Kreuzberg Berlin,10997,Kreuzberg,Berlin,no,Studio,", 10997 Kreuzberg Berlin"
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,Kaulsdorf Berlin,12627,Kaulsdorf,Berlin,yes,Studio,", 12627 Kaulsdorf Berlin"


In [15]:
import time
from tqdm import tqdm # Import this


def geocode_address_column(df, address_column, sleep_between=1, user_agent="geo_project"):
    geolocator = Nominatim(user_agent=user_agent, timeout=20)

    lats, lons = [], []
    
    # Wrap the iterator with tqdm for a progress bar
    for addr in tqdm(df[address_column], desc="Geocoding addresses"):
        try:
            location = geolocator.geocode(addr)
            if location:
                lats.append(location.latitude)
                lons.append(location.longitude)
            else:
                lats.append(None)
                lons.append(None)
        except Exception:
            lats.append(None)
            lons.append(None)
        
        time.sleep(sleep_between)

    df['latitude'] = lats
    df['longitude'] = lons
    return df

In [17]:
data = geocode_address_column(data, address_column='address', sleep_between=1)

Geocoding addresses: 100%|██████████| 5344/5344 [2:56:56<00:00,  1.99s/it]  


In [34]:
data.isna().sum()

detail_url            0
raw_info              0
type                 12
price_euro         1083
number_of_rooms    1101
surface_m2         1108
floor              1732
street             4251
house_number       4251
neighborhood       1083
postal_code           0
region             1083
city               1072
first_tenant          0
type_normalized      12
address               0
latitude            934
longitude           934
dtype: int64

In [35]:
def df_to_gdf_points(df, lon_col, lat_col):
    """
    Converts dataframe with lon/lat columns to GeoDataFrame with Point geometry.
    """
    df = df.dropna(subset=[lon_col, lat_col])
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

addresses_gdf = df_to_gdf_points(data, lon_col='longitude', lat_col='latitude')

In [36]:
addresses_gdf.isna().sum()

detail_url            0
raw_info              0
type                 12
price_euro         1083
number_of_rooms    1098
surface_m2         1085
floor              1708
street             4250
house_number       4250
neighborhood       1083
postal_code           0
region             1083
city               1072
first_tenant          0
type_normalized      12
address               0
latitude              0
longitude             0
geometry              0
dtype: int64

In [37]:
addresses_gdf

,detail_url,raw_info,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,neighborhood,postal_code,region,city,first_tenant,type_normalized,address,latitude,longitude,geometry
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.52055 52.50294)
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,Friedrichsfelde Berlin,10315,Friedrichsfelde,Berlin,yes,Wohnung,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.52055 52.50294)
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,Friedenau Berlin,12159,Friedenau,Berlin,no,Wohnung,", 12159 Friedenau Berlin",52.472915,13.336917,POINT (13.33692 52.47291)
5,https://www.immowelt.de/expose/bc30a029-9da4-4...,"Wohnung zur Miete 1.399 € 4 Zimmer 90,5 m² 1. ...",Wohnung,1399.0,4.0,90.5,1,NaN,NaN,Wilhelmshagen Berlin,12589,Wilhelmshagen,Berlin,no,Wohnung,", 12589 Wilhelmshagen Berlin",52.438492,13.722329,POINT (13.72233 52.43849)
6,https://www.immowelt.de/expose/a2fc0059-8d75-4...,Studio zur Miete 440 € 1 Zimmer 10 m² 5. Gesch...,Studio,440.0,1.0,10.0,5,NaN,NaN,Oberschöneweide Berlin,12459,Oberschöneweide,Berlin,no,Studio,", 12459 Oberschöneweide Berlin",52.468305,13.514753,POINT (13.51475 52.4683)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,Spandau Berlin,13591,Spandau,Berlin,no,Studio,", 13591 Spandau Berlin",52.536515,13.139829,POINT (13.13983 52.53652)
5340,https://www.immowelt.de/expose/5a4ba852-5719-4...,Wohnungsswap.de,Wohnung,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,no,Wohnung,", <NA>",-23.233550,17.323111,POINT (17.32311 -23.23355)
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,Kreuzberg Berlin,10997,Kreuzberg,Berlin,no,Studio,", 10997 Kreuzberg Berlin",52.501353,13.433080,POINT (13.43308 52.50135)
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,Kaulsdorf Berlin,12627,Kaulsdorf,Berlin,yes,Studio,", 12627 Kaulsdorf Berlin",52.512135,13.590192,POINT (13.59019 52.51214)


In [38]:
neighborhoods = gpd.read_file("/Users/kajetanhanausek/Desktop/cloned_repos/layered-populate-data-pool-da/mapping/lor_ortsteile.geojson")

In [39]:
neighborhoods

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,0103,Hansaviertel,Polygon,Hansaviertel,Mitte,52.5337,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,0104,Tiergarten,Polygon,Tiergarten,Mitte,516.0672,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,0105,Wedding,Polygon,Wedding,Mitte,919.9112,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."
...,...,...,...,...,...,...,...,...
91,re_ortsteil.1207,1207,Waidmannslust,Polygon,Waidmannslust,Reinickendorf,223.7780,"POLYGON ((13.33106 52.61487, 13.33097 52.61479..."
92,re_ortsteil.1208,1208,Lübars,Polygon,Lübars,Reinickendorf,499.1961,"POLYGON ((13.36693 52.62535, 13.36692 52.62533..."
93,re_ortsteil.1209,1209,Wittenau,Polygon,Wittenau,Reinickendorf,587.1148,"POLYGON ((13.34413 52.58284, 13.34414 52.58285..."
94,re_ortsteil.1210,1210,Märkisches Viertel,Polygon,Märkisches Viertel,Reinickendorf,323.9201,"POLYGON ((13.34756 52.59008, 13.34757 52.59009..."


In [40]:
addresses_gdf = gpd.sjoin(
    addresses_gdf,
    neighborhoods[['geometry','spatial_name', 'OTEIL', 'BEZIRK']],
    how='left', predicate='within'
)

In [29]:
addresses_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 4410 entries, 0 to 5343
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   detail_url       4410 non-null   object  
 1   raw_info         4410 non-null   object  
 2   type             4398 non-null   object  
 3   price_euro       3327 non-null   float64 
 4   number_of_rooms  3312 non-null   float64 
 5   surface_m2       3325 non-null   float64 
 6   floor            2702 non-null   object  
 7   street           160 non-null    object  
 8   house_number     160 non-null    object  
 9   neighborhood     3327 non-null   object  
 10  postal_code      4410 non-null   object  
 11  region           3327 non-null   object  
 12  city             3338 non-null   object  
 13  first_tenant     4410 non-null   object  
 14  type_normalized  4398 non-null   object  
 15  address          4410 non-null   object  
 16  latitude         4410 non-null   float6

In [71]:
addresses_gdf = addresses_gdf.drop(columns=['index_right', 'city', 'region', 'type_normalized', 'neighborhood'])

KeyError: "['index_right', 'city', 'region', 'type_normalized'] not found in axis"

In [106]:
addresses_gdf = addresses_gdf.drop(columns=['neighborhood'])

KeyError: "['neighborhood'] not found in axis"

In [107]:
listings = addresses_gdf.copy()

In [108]:
listings['geometry']=listings['geometry'].apply(lambda x: str(x))

In [109]:
listings.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 4410 entries, 0 to 5343
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   detail_url       4410 non-null   object 
 1   raw_info         4410 non-null   object 
 2   type             4398 non-null   object 
 3   price_euro       3327 non-null   float64
 4   number_of_rooms  3312 non-null   float64
 5   surface_m2       3325 non-null   float64
 6   floor            2702 non-null   object 
 7   street           160 non-null    object 
 8   house_number     160 non-null    object 
 9   postal_code      4410 non-null   object 
 10  first_tenant     4410 non-null   object 
 11  address          4410 non-null   object 
 12  latitude         4410 non-null   float64
 13  longitude        4410 non-null   float64
 14  geometry         4410 non-null   object 
 15  spatial_name     3337 non-null   object 
 16  OTEIL            3337 non-null   object 
 17  BEZIRK     

In [110]:
listings

,detail_url,raw_info,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,postal_code,first_tenant,address,latitude,longitude,geometry,spatial_name,OTEIL,BEZIRK
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,12159,no,", 12159 Friedenau Berlin",52.472915,13.336917,POINT (13.3369173 52.4729146),0702,Friedenau,Tempelhof-Schöneberg
5,https://www.immowelt.de/expose/bc30a029-9da4-4...,"Wohnung zur Miete 1.399 € 4 Zimmer 90,5 m² 1. ...",Wohnung,1399.0,4.0,90.5,1,NaN,NaN,12589,no,", 12589 Wilhelmshagen Berlin",52.438492,13.722329,POINT (13.7223287 52.4384919),0912,Rahnsdorf,Treptow-Köpenick
6,https://www.immowelt.de/expose/a2fc0059-8d75-4...,Studio zur Miete 440 € 1 Zimmer 10 m² 5. Gesch...,Studio,440.0,1.0,10.0,5,NaN,NaN,12459,no,", 12459 Oberschöneweide Berlin",52.468305,13.514753,POINT (13.5147534 52.468305),0909,Oberschöneweide,Treptow-Köpenick
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,13591,no,", 13591 Spandau Berlin",52.536515,13.139829,POINT (13.1398288 52.536515),0504,Staaken,Spandau
5340,https://www.immowelt.de/expose/5a4ba852-5719-4...,Wohnungsswap.de,Wohnung,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,no,", <NA>",-23.233550,17.323111,POINT (17.3231107 -23.2335499),NaN,NaN,NaN
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,10997,no,", 10997 Kreuzberg Berlin",52.501353,13.433080,POINT (13.4330798 52.5013525),0202,Kreuzberg,Friedrichshain-Kreuzberg
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,12627,yes,", 12627 Kaulsdorf Berlin",52.512135,13.590192,POINT (13.5901922 52.5121351),1003,Kaulsdorf,Marzahn-Hellersdorf


In [111]:
listings = listings.rename(columns={
    'raw_info': 'name',
    'spatial_name': 'neighborhood_id',
    'OTEIL': 'neighborhood',
    'BEZIRK': 'district'
})

In [112]:
listings.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 4410 entries, 0 to 5343
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   detail_url       4410 non-null   object 
 1   name             4410 non-null   object 
 2   type             4398 non-null   object 
 3   price_euro       3327 non-null   float64
 4   number_of_rooms  3312 non-null   float64
 5   surface_m2       3325 non-null   float64
 6   floor            2702 non-null   object 
 7   street           160 non-null    object 
 8   house_number     160 non-null    object 
 9   postal_code      4410 non-null   object 
 10  first_tenant     4410 non-null   object 
 11  address          4410 non-null   object 
 12  latitude         4410 non-null   float64
 13  longitude        4410 non-null   float64
 14  geometry         4410 non-null   object 
 15  neighborhood_id  3337 non-null   object 
 16  neighborhood     3337 non-null   object 
 17  district   

In [113]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
listings['district_id'] = listings['district'].map(district_mapping).astype(str)

In [114]:
listings.isna().sum()


detail_url            0
name                  0
type                 12
price_euro         1083
number_of_rooms    1098
surface_m2         1085
floor              1708
street             4250
house_number       4250
postal_code           0
first_tenant          0
address               0
latitude              0
longitude             0
geometry              0
neighborhood_id    1073
neighborhood       1073
district           1073
district_id           0
dtype: int64

In [115]:
listings.describe()

,price_euro,number_of_rooms,surface_m2,latitude,longitude
count,3327.000000,3312.000000,3325.000000,4410.000000,4410.000000
mean,995.656748,2.348430,68.921955,34.096448,14.357711
std,846.310067,0.925235,35.305367,32.492697,1.682794
min,3.000000,1.000000,4.000000,-23.233550,13.139829
25%,530.000000,2.000000,50.000000,52.411811,13.373084
50%,770.000000,2.000000,64.000000,52.489606,13.432282
75%,1200.000000,3.000000,81.000000,52.530125,13.664349
max,16500.000000,8.000000,750.000000,52.980194,17.323111


In [116]:
listings = listings.dropna(subset=['district'])

In [117]:
listings.describe()

,price_euro,number_of_rooms,surface_m2,latitude,longitude
count,3326.000000,3311.000000,3324.000000,3337.000000,3337.000000
mean,993.851473,2.347931,68.886131,52.507857,13.405207
std,840.006031,0.924929,35.250187,0.047987,0.096322
min,3.000000,1.000000,4.000000,52.383290,13.139829
25%,530.000000,2.000000,50.000000,52.473187,13.348785
50%,770.000000,2.000000,64.000000,52.511632,13.410326
75%,1200.000000,3.000000,80.925000,52.541084,13.451379
max,16500.000000,8.000000,750.000000,52.639901,13.722329


In [118]:
listings.isna().sum()

detail_url            0
name                  0
type                  5
price_euro           11
number_of_rooms      26
surface_m2           13
floor               636
street             3177
house_number       3177
postal_code           0
first_tenant          0
address               0
latitude              0
longitude             0
geometry              0
neighborhood_id       0
neighborhood          0
district              0
district_id           0
dtype: int64

In [119]:
listings

,detail_url,name,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,postal_code,first_tenant,address,latitude,longitude,geometry,neighborhood_id,neighborhood,district,district_id
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg,11011011
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg,11011011
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,12159,no,", 12159 Friedenau Berlin",52.472915,13.336917,POINT (13.3369173 52.4729146),0702,Friedenau,Tempelhof-Schöneberg,11007007
5,https://www.immowelt.de/expose/bc30a029-9da4-4...,"Wohnung zur Miete 1.399 € 4 Zimmer 90,5 m² 1. ...",Wohnung,1399.0,4.0,90.5,1,NaN,NaN,12589,no,", 12589 Wilhelmshagen Berlin",52.438492,13.722329,POINT (13.7223287 52.4384919),0912,Rahnsdorf,Treptow-Köpenick,11009009
6,https://www.immowelt.de/expose/a2fc0059-8d75-4...,Studio zur Miete 440 € 1 Zimmer 10 m² 5. Gesch...,Studio,440.0,1.0,10.0,5,NaN,NaN,12459,no,", 12459 Oberschöneweide Berlin",52.468305,13.514753,POINT (13.5147534 52.468305),0909,Oberschöneweide,Treptow-Köpenick,11009009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5337,https://www.immowelt.de/expose/d8478187-6338-4...,Studio zur Miete Tauschwohnung 400 € 3 Zimmer ...,Studio,400.0,3.0,72.0,4,NaN,NaN,10969,no,", 10969 Kreuzberg Berlin",52.502354,13.402346,POINT (13.4023459 52.5023538),0202,Kreuzberg,Friedrichshain-Kreuzberg,11002002
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,13591,no,", 13591 Spandau Berlin",52.536515,13.139829,POINT (13.1398288 52.536515),0504,Staaken,Spandau,11005005
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,10997,no,", 10997 Kreuzberg Berlin",52.501353,13.433080,POINT (13.4330798 52.5013525),0202,Kreuzberg,Friedrichshain-Kreuzberg,11002002
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,12627,yes,", 12627 Kaulsdorf Berlin",52.512135,13.590192,POINT (13.5901922 52.5121351),1003,Kaulsdorf,Marzahn-Hellersdorf,11010010


In [120]:
import pandas as pd
from datetime import datetime

# 1. Get today's date in a clean format (e.g., 20240522)
date_prefix = datetime.now().strftime('%Y%m%d')

# 2. Generate a range of numbers starting from 10,000
# len(addresses_gdf) ensures we have exactly one ID per row
start_val = 10000
id_range = range(start_val, start_val + len(listings))

# 3. Create the column (combining date + number as a string)
listings['id'] = [f"{date_prefix}_{i}" for i in id_range]

print(listings[['id']].head())

               id
0  20260206_10000
1  20260206_10001
3  20260206_10002
5  20260206_10003
6  20260206_10004


In [121]:
listings

,detail_url,name,type,price_euro,number_of_rooms,surface_m2,floor,street,house_number,postal_code,first_tenant,address,latitude,longitude,geometry,neighborhood_id,neighborhood,district,district_id,id
0,https://www.immowelt.de/expose/3e6739d3-ced0-4...,Wohnung zur Miete - Erstbezug 1.390 € 2 Zimmer...,Wohnung,1390.0,2.0,55.7,1,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg,11011011,20260206_10000
1,https://www.immowelt.de/expose/572f03ad-a183-4...,Wohnung zur Miete - Erstbezug 2.290 € 4 Zimmer...,Wohnung,2290.0,4.0,107.5,0,NaN,NaN,10315,yes,", 10315 Friedrichsfelde Berlin",52.502936,13.520546,POINT (13.5205456 52.5029358),1101,Friedrichsfelde,Lichtenberg,11011011,20260206_10001
3,https://www.immowelt.de/expose/499034c0-a638-4...,"Wohnung zur Miete 929 € 1 Zimmer 34,4 m² 1. Ge...",Wohnung,929.0,1.0,34.4,1,NaN,NaN,12159,no,", 12159 Friedenau Berlin",52.472915,13.336917,POINT (13.3369173 52.4729146),0702,Friedenau,Tempelhof-Schöneberg,11007007,20260206_10002
5,https://www.immowelt.de/expose/bc30a029-9da4-4...,"Wohnung zur Miete 1.399 € 4 Zimmer 90,5 m² 1. ...",Wohnung,1399.0,4.0,90.5,1,NaN,NaN,12589,no,", 12589 Wilhelmshagen Berlin",52.438492,13.722329,POINT (13.7223287 52.4384919),0912,Rahnsdorf,Treptow-Köpenick,11009009,20260206_10003
6,https://www.immowelt.de/expose/a2fc0059-8d75-4...,Studio zur Miete 440 € 1 Zimmer 10 m² 5. Gesch...,Studio,440.0,1.0,10.0,5,NaN,NaN,12459,no,", 12459 Oberschöneweide Berlin",52.468305,13.514753,POINT (13.5147534 52.468305),0909,Oberschöneweide,Treptow-Köpenick,11009009,20260206_10004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5337,https://www.immowelt.de/expose/d8478187-6338-4...,Studio zur Miete Tauschwohnung 400 € 3 Zimmer ...,Studio,400.0,3.0,72.0,4,NaN,NaN,10969,no,", 10969 Kreuzberg Berlin",52.502354,13.402346,POINT (13.4023459 52.5023538),0202,Kreuzberg,Friedrichshain-Kreuzberg,11002002,20260206_13332
5339,https://www.immowelt.de/expose/535d40da-cf9a-4...,Studio zur Miete Tauschwohnung 285 € 1 Zimmer ...,Studio,285.0,1.0,38.0,4,NaN,NaN,13591,no,", 13591 Spandau Berlin",52.536515,13.139829,POINT (13.1398288 52.536515),0504,Staaken,Spandau,11005005,20260206_13333
5341,https://www.immowelt.de/expose/a8cf0052-2df0-4...,Studio zur Miete Tauschwohnung 390 € 2 Zimmer ...,Studio,390.0,2.0,42.0,3,NaN,NaN,10997,no,", 10997 Kreuzberg Berlin",52.501353,13.433080,POINT (13.4330798 52.5013525),0202,Kreuzberg,Friedrichshain-Kreuzberg,11002002,20260206_13334
5342,https://www.immowelt.de/expose/a64cad53-e445-4...,Studio zur Miete - Erstbezug 1.233 € 1 Zimmer ...,Studio,1233.0,1.0,12.0,1,NaN,NaN,12627,yes,", 12627 Kaulsdorf Berlin",52.512135,13.590192,POINT (13.5901922 52.5121351),1003,Kaulsdorf,Marzahn-Hellersdorf,11010010,20260206_13335


In [122]:
listings.to_csv('listings_final.csv')

In [132]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")


engine = create_engine('postgresql+psycopg2://testing:z&e8AqIt#HE>Lks[1}Go@localhost:5433/layereddb')

query_access=f"""
DROP TABLE berlin_source_data.long_term_listings;

CREATE TABLE berlin_source_data.long_term_listings (
	id varchar NOT NULL,
	detail_url varchar NULL,
	"name" varchar NULL,
	"type" varchar NULL,
	first_tenant varchar NULL,
	price_euro int4 NULL,
	number_of_rooms float8 NULL,
	surface_m2 float8 NULL,
	floor varchar NULL,
	street varchar NULL,
	house_number varchar NULL,
	neighborhood varchar NULL,
	district varchar NULL,
	postal_code int4 NULL,
	city varchar NULL,
	address text NULL,
	latitude numeric(9, 6) NULL,
	longitude numeric(9, 6) NULL,
	geometry varchar NULL,
	district_id varchar NOT NULL,
	neighborhood_id varchar(20) NULL,
	CONSTRAINT long_term_listings_pkey PRIMARY KEY (id),
	CONSTRAINT district_id_fk FOREIGN KEY (district_id) REFERENCES berlin_source_data.districts(district_id) ON DELETE RESTRICT ON UPDATE CASCADE
);

"""
with engine.connect() as conn:
    conn.execute(text(query_access))
    conn.commit()  # commit the transaction

In [133]:
import numpy as np

# 1. Convert any nullable pandas types to standard object types 
# This breaks the "Int64" / "string" logic that forces the <NA> behavior
listings = listings.astype(object)

# 2. Replace all forms of 'nothing' with None
# This covers numpy NaN, pandas NA, and the literal string "<NA>" 
listings = listings.where(pd.notnull(listings), None)
listings = listings.replace("<NA>", None)

# 3. Export again
listings.to_sql(
    name='long_term_listings',
    con=engine, 
    schema='berlin_source_data',
    if_exists='append', 
    index=False
)


337

In [134]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3337 entries, 0 to 5343
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   detail_url       3337 non-null   object
 1   name             3337 non-null   object
 2   type             3332 non-null   object
 3   price_euro       3326 non-null   object
 4   number_of_rooms  3311 non-null   object
 5   surface_m2       3324 non-null   object
 6   floor            2701 non-null   object
 7   street           160 non-null    object
 8   house_number     160 non-null    object
 9   postal_code      3326 non-null   object
 10  first_tenant     3337 non-null   object
 11  address          3337 non-null   object
 12  latitude         3337 non-null   object
 13  longitude        3337 non-null   object
 14  geometry         3337 non-null   object
 15  neighborhood_id  3337 non-null   object
 16  neighborhood     3337 non-null   object
 17  district         3337 non-null   objec